# Create z-score files

This notebook creates mean/std files for the paper. Resdidual coefficients are defined in a separated notebook.

In [1]:
import os
import yaml
import numpy as np
import xarray as xr

## ERA5 mean std

In [2]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_ERA5.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [3]:
N_levels = 6

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/ERA5_1h_8km/'
ds_example = xr.open_zarr(base_dir+'ERA5_FULL_1h_2000.zarr')
level = np.array(ds_example['level'])

In [4]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_surf = list(set(varnames) - set(['U', 'V', 'T', 'Q']))
varname_upper = ['U', 'V', 'T', 'Q']

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [5]:
varnames

['U', 'V', 'T', 'Q', 'SP', 'MSL', 'VAR_2T', 'VAR_10U', 'VAR_10V', 'PWAT_05']

In [6]:
varname_surf

['VAR_2T', 'VAR_10V', 'MSL', 'SP', 'VAR_10U', 'PWAT_05']

In [7]:
conf['zscore']['save_loc']

'/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_FULL/temp_ERA5_npy/'

### Mean file

In [8]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean = xr.Dataset(coords={"level": level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_mean[varname] = data_array
    else:
        data_array = xr.DataArray(data, name=varname,)
        ds_mean[varname] = data_array

In [9]:
ds_mean.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/ERA5_mean_1980_2019.nc')

In [10]:
ds_GP = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_3h_mean_1980_2019.nc')
ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/ERA5_mean_1980_2019.nc')

for varname in ds_full.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_full[varname].values)
    except:
        pass

=================== VAR_2T ===================
290.3522173722694
286.2614150433643
=================== VAR_10V ===================
0.873942350769839
-0.2543528365298459
=================== MSL ===================
101636.02926621184
101589.02650303871
=================== SP ===================
=================== VAR_10U ===================
-0.26072289415013494
0.3811609779844044
=================== PWAT_05 ===================
=================== U ===================
[-0.34319156  0.03790973  2.74288486  6.42386911  8.82248812 11.59287245
 14.94868826 19.52588245 24.8600006  14.33904524  0.73781285]
[ 0.72124789  1.44119645  2.30388616  5.41507396 10.1803159  21.18231383]
=================== V ===================
[ 0.89797592  1.94231892  2.17507869  0.39714699 -0.05320645 -0.00847903
  0.31194102  0.89837034  1.34006671  0.52205376 -0.09927631]
[-0.12261816  0.03269744  0.01887282 -0.23605474 -0.56320148 -0.26393257]
=================== T ===================
[292.18098043 289.54826956

### Std file

In [11]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std = xr.Dataset(coords={"level": level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["level",],
            coords={"level": level},
            name=varname,
        )
        ds_std[varname] = data_array
    else:
        data_array = xr.DataArray(data, name=varname)
        ds_std[varname] = data_array

In [12]:
ds_std.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/ERA5_std_1980_2019.nc', mode='w')

In [13]:
ds_GP = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/ERA5_3h_std_1980_2019.nc')
ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/ERA5_std_1980_2019.nc')

for varname in ds_full.keys():
    print(f'=================== {varname} ===================')
    try:
        print(ds_GP[varname].values)
        print(ds_full[varname].values)
    except:
        pass

=================== VAR_2T ===================
10.253738201019486
12.400036205768435
=================== VAR_10V ===================
3.377895893264416
3.8345185032550857
=================== MSL ===================
669.4589901534357
714.3823301720362
=================== SP ===================
=================== VAR_10U ===================
2.3918602602295924
3.6321173036915293
=================== PWAT_05 ===================
=================== U ===================
[ 2.63879745  4.44049216  5.8813855   7.1103909   8.7521183  10.6184807
 12.99767326 16.19460319 18.32331105 12.12329474  8.48019539]
[ 5.18526331  5.82218562  6.1607335   7.66685534 10.98991346 16.9186046 ]
=================== V ===================
[ 3.68121891  6.24882356  7.73042307  7.41146002  8.34179071  9.70744222
 11.67821738 14.3481787  14.83903406  7.16725947  3.41170739]
[ 5.66619223  6.24511802  6.3450902   7.25532646 10.26387914 15.56777731]
=================== T ===================
[9.73946451 9.48952678 7.84529

## WRF mean std

In [2]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_WRF.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [8]:
N_levels = 12

base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/C404/'
ds_example = xr.open_zarr(base_dir+'C404_FULL_2000.zarr')
level = np.array(ds_example['bottom_top'])

In [9]:
varnames = list(conf['zscore'].keys())
varnames = varnames[:-3] # remove save_loc and others
varname_upper = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_P', 'WRF_Q_tot', 'WRF_Q_tot_05', 'WRF_W', 'WRF_Z']
varname_surf = list(set(varnames) - set(varname_upper))

# collect computed mean and variance values
# See "qsub_STEP01_compute_mean_std.ipynb"
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['zscore']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['zscore']['prefix'], varname)
    
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['zscore']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['zscore']['prefix'], i_level, varname)
        
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [10]:
# ------------------------------------------------------- #
# Initialize dataset
ds_mean = xr.Dataset(coords={'bottom_top': level})

for varname, data in MEAN_values.items():
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_mean[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_mean[varname] = data_array

In [13]:
# ds_mean.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_mean_1980_2019_12lev.nc')

In [15]:
# ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_mean_1980_2019_12lev.nc')
# ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_mean_1980_2019_12lev.nc')

# for varname in ds_full.keys():
#     print(f'=================== {varname} ===================')
#     try:
#         print(ds_full[varname].values)
#         print(ds_new [varname].values)
#     except:
#         pass

In [17]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std = xr.Dataset(coords={'bottom_top': level})

for varname, data in STD_values.items():
    data = np.sqrt(data)
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["bottom_top",],
            coords={"bottom_top": level},
            name=varname,
        )
        ds_std[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std[varname] = data_array

In [18]:
# ds_std.to_netcdf('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_std_1980_2019_12lev.nc')

In [14]:
# ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_std_1980_2019_12lev.nc')
# ds_full = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_FULL/mean_std/C404_std_1980_2019_12lev.nc')

# for varname in ds_full.keys():
#     print(f'=================== {varname} ===================')
#     try:
#         print(ds_full[varname].values)
#         print(ds_new [varname].values)
#     except:
#         pass